# 02 — SetFit on BANKING77

一个方法、一个数据集；从项目根目录或 notebooks/ 顺序运行。所有函数内联，无辅助 .py 或审计 CSV/JSON 依赖。公开数据、模型使用缓存或在线加载；CLINC 原始 JSON 缺失时在内存中下载并校验 SHA256。

默认使用全量清理训练池（SHOT_BUDGET=None），仅评估 Validation（EVALUATION_SPLIT="validation", FINAL_TEST=False）。训练参数、协议、Label 编码与计时方式保持原有设置。build_protocol 与 01/02/03 逐行一致，修改需同步。

结果、配置、版本、预测及可读错误示例直接显示在单元中，不导出结果 CSV/JSON；Trainer 的临时工作目录自动清理，不保存权重。完整预测保存在内存 summary["predictions"]，完整错误保存在 summary["errors"]。

最终提交时另存为 Name_ID_SetFit_BANKING77.ipynb，运行全部单元并保留输出。只将已完成实验的 Notebook 加入 Name_ID_source.zip；当前开发文件名无需立刻更名。

In [1]:
# Set True only when explicitly choosing to install missing packages in this interpreter.
INSTALL_MISSING_DEPENDENCIES = False
from importlib import util, metadata
import subprocess
import sys

_requirements = {"numpy": "numpy", "pandas": "pandas", "datasets": "datasets",
                 "torch": "torch", "setfit": "setfit", "sentence_transformers": "sentence-transformers",
                 "transformers": "transformers", "sklearn": "scikit-learn"}
_missing = [package for module, package in _requirements.items() if util.find_spec(module) is None]
if _missing:
    print("Missing dependencies:", _missing)
    if not INSTALL_MISSING_DEPENDENCIES:
        raise RuntimeError("Install in the current kernel: " + sys.executable + " -m pip install " + " ".join(_missing) + "; or set INSTALL_MISSING_DEPENDENCIES=True and rerun setup")
    subprocess.check_call([sys.executable, "-m", "pip", "install", *_missing])

import hashlib
import json
import re
from collections import Counter, defaultdict
from pathlib import Path

import pandas as pd
from datasets import load_dataset
from IPython.display import display
from sklearn.model_selection import train_test_split

# Resolve output paths without inspecting another notebook or results files.
cwd = Path.cwd()
PROJECT = cwd.parent if cwd.name == "notebooks" else cwd

import random
import time
from tempfile import TemporaryDirectory
from importlib import metadata

import numpy as np
import torch
from datasets import Dataset
from setfit import SetFitModel, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score

DATASET = "banking77"
SHOT_BUDGET = None  # Full cleaned pool; set to 8 for an optional low-resource condition.
SEED = 42
EVALUATION_SPLIT = "validation"
FINAL_TEST = False
VALIDATION_FRACTION = 0.10
BACKBONE = "sentence-transformers/all-MiniLM-L6-v2"
BATCH_SIZE = 16
NUM_EPOCHS = 1
NUM_ITERATIONS = 5
BODY_LEARNING_RATE = 2e-5
RUN_SHOT_GRID = False
VERSIONS = {package: metadata.version(package) for package in
            ["torch", "setfit", "sentence-transformers", "transformers", "datasets", "scikit-learn"]}
print(VERSIONS)


# P1: sampling and optional repeated-seed controlled analysis.
SAMPLING_MODE = "independent"
SEEDS = [42, 43, 44]
RUN_MULTI_SEED = False
ANALYSIS_DATASETS = ["banking77"]
ANALYSIS_SAMPLING_MODES = ["independent", "nested"]
CLINC_SHA256 = "36923c3705a59e08fe9c3883d8bc2dd966ef93e22cb78ac41171782a698d56e0"
NORMALIZATION = "strip/collapse whitespace and Unicode casefold; punctuation retained"


def normalize_text(text):
    if not isinstance(text, str) or not text.strip():
        raise ValueError("Text must be a non-empty string")
    return re.sub(r'\s+', ' ', text).strip().casefold()


def text_hash(text):
    return hashlib.sha256(normalize_text(text).encode("utf-8")).hexdigest()


def audit_overlap(left, right, left_name, right_name):
    a = left.assign(_key=left["text"].map(normalize_text))
    b = right.assign(_key=right["text"].map(normalize_text))
    # Keep the original audit column order for the saved 01 evidence.
    pairs = a.drop(columns="text_sha256", errors="ignore").merge(
        b.drop(columns="text_sha256", errors="ignore"),
        on="_key", suffixes=("_left", "_right")
    )
    report = {
        "left": left_name, "right": right_name,
        "matching_pairs": len(pairs), "unique_texts": int(pairs["_key"].nunique()),
        "left_rows": int(pairs["source_id_left"].nunique()),
        "right_rows": int(pairs["source_id_right"].nunique()),
        "conflicting_pairs": int((pairs["label_left"] != pairs["label_right"]).sum()),
    }
    return report, pairs


def remove_train_overlap(train, protected, name):
    report, pairs = audit_overlap(train, protected, name, "protected")
    drop_ids = set(pairs["source_id_left"])
    cleaned = train.loc[~train["source_id"].isin(drop_ids)].copy()
    assert len(train) - len(cleaned) == report["left_rows"]
    print(name, "removed training rows:", len(drop_ids),
          "conflicting pairs:", report["conflicting_pairs"])
    return cleaned, report, pairs


def verify_disjoint(a, b, name_a, name_b):
    overlap = set(a["text_sha256"]) & set(b["text_sha256"])
    assert not overlap, f"Normalized-text overlap: {name_a} / {name_b}"
    assert set(a["source_id"]).isdisjoint(b["source_id"])
    return {"left": name_a, "right": name_b, "unique_texts": len(overlap)}


def make_source_ids(df, dataset_name, split_name):
    prefix = "banking77" if dataset_name == "banking77" else "clinc"
    result = df.copy()
    result["source_id"] = [f"{prefix}_{split_name}_{i}" for i in range(len(result))]
    result["text_sha256"] = result["text"].map(text_hash)
    return result


def read_original_clinc():
    """Read optional local original data, or download in memory; never export JSON."""
    path = PROJECT / "data" / "clinc150" / "data_full.json"
    if path.exists():
        content = path.read_bytes()
    else:
        import urllib.request
        url = "https://raw.githubusercontent.com/clinc/oos-eval/master/data/data_full.json"
        content = urllib.request.urlopen(url, timeout=90).read()
    assert hashlib.sha256(content).hexdigest() == CLINC_SHA256, "Original CLINC SHA256 mismatch"
    return json.loads(content)


def sample_fewshot(pool, shot_budget, seed, sampling_mode):
    """Sample a few-shot set; nested budgets share per-class random prefixes."""
    if sampling_mode not in {"independent", "nested"}:
        raise ValueError("sampling_mode must be 'independent' or 'nested'")
    if shot_budget is None:
        return pool.copy()
    if type(shot_budget) is not int or shot_budget <= 0:
        raise ValueError("shot_budget must be None or a positive integer")
    if sampling_mode == "nested" and shot_budget > 16:
        raise ValueError("nested sampling supports shot budgets up to 16")
    assert pool.groupby("label").size().min() >= shot_budget, "Insufficient examples per class"
    if sampling_mode == "independent":
        # Preserve legacy pandas sample behavior, IDs and row order exactly.
        return pool.groupby("label", group_keys=False).sample(
            n=shot_budget, random_state=seed).copy()

    rng = np.random.RandomState(seed)
    chosen_ids = []
    for label in sorted(pool["label"].unique().tolist()):
        class_ids = pool.loc[pool["label"] == label, "source_id"].to_numpy()
        chosen_ids.extend(rng.permutation(class_ids)[:shot_budget].tolist())
    selected_ids = set(chosen_ids)
    return pool.loc[pool["source_id"].isin(selected_ids)].copy()


def build_protocol(dataset_name, shot_budget=8, seed=42, validation_fraction=0.10,
                   sampling_mode="independent"):
    if dataset_name not in {"banking77", "clinc150"}:
        raise ValueError("Choose banking77 or clinc150")
    if shot_budget is not None and (type(shot_budget) is not int or shot_budget <= 0):
        raise ValueError("shot_budget must be None or a positive integer")
    if sampling_mode not in {"independent", "nested"}:
        raise ValueError("sampling_mode must be 'independent' or 'nested'")
    if sampling_mode == "nested" and shot_budget is not None and shot_budget > 16:
        raise ValueError("nested sampling supports shot budgets up to 16")
    if type(seed) is not int or not 0 <= seed < 2**32:
        raise ValueError("seed must be an integer in [0, 2**32)")
    if not 0 < validation_fraction < 1:
        raise ValueError("validation_fraction must be between zero and one")
    audits = {}
    pairs = {}
    oos_audit = []
    original_sha256 = None
    if dataset_name == "banking77":
        raw = load_dataset("PolyAI/banking77")
        id_to_name = dict(enumerate(raw["train"].features["label"].names))
        expected_classes = 77
        train = make_source_ids(raw["train"].to_pandas(), dataset_name, "train")
        test = make_source_ids(raw["test"].to_pandas(), dataset_name, "test")
        cleaned, audits["train_vs_test"], pairs["train_vs_test"] = remove_train_overlap(train, test, "banking_train")
        pool, validation = train_test_split(
            cleaned, test_size=validation_fraction, random_state=seed,
            stratify=cleaned["label"]
        )
        pool, audits["pool_vs_validation"], pairs["pool_vs_validation"] = remove_train_overlap(pool, validation, "banking_pool")
        original_splits = {"train": train, "test": test}
        expected_labels = set(id_to_name)
    else:
        original = read_original_clinc()
        original_sha256 = CLINC_SHA256
        converted = load_dataset("DeepPavlov/clinc150", "default")
        id_to_names = defaultdict(set)
        original_splits = {}
        all_original_texts = {text for rows in original.values() for text, label in rows}
        for split, raw_name in [("train", "train"), ("validation", "val"), ("test", "test")]:
            raw_rows = original[raw_name]
            text_to_names = defaultdict(set)
            for text, name in raw_rows:
                text_to_names[text].add(name)
            assert all(len(names) == 1 for names in text_to_names.values()), "Ambiguous original labels"
            hf_frame = converted[split].to_pandas()
            known = hf_frame.loc[hf_frame["label"].notna()]
            assert Counter(known["utterance"]) == Counter(text for text, name in raw_rows)
            for text, number in zip(known["utterance"], known["label"]):
                id_to_names[int(number)].update(text_to_names[text])
            missing = Counter(hf_frame.loc[hf_frame["label"].isna(), "utterance"])
            oos = Counter(text for text, name in original["oos_" + raw_name])
            unknown = missing - oos
            assert not (oos - missing), "Original OOS rows missing from converted split"
            oos_audit.append({
                "split": split, "null_labels": sum(missing.values()),
                "original_oos_matches": sum((missing & oos).values()),
                "unconfirmed_extra_rows": sum(unknown.values()),
                "extra_rows_in_any_original_split": sum(n for text, n in unknown.items() if text in all_original_texts),
            })
            frame = pd.DataFrame(raw_rows, columns=["text", "label"])
            # Preserve the established IDs: i indexes original JSON in-domain rows.
            original_splits[split] = make_source_ids(frame, dataset_name, split)
        assert len(id_to_names) == 150 and all(len(names) == 1 for names in id_to_names.values())
        id_to_name = {number: next(iter(names)) for number, names in id_to_names.items()}
        assert set(id_to_name) == set(range(150)) and len(set(id_to_name.values())) == 150
        expected_classes = 150
        expected_labels = set(id_to_name.values())
        train = original_splits["train"]
        validation = original_splits["validation"]
        test = original_splits["test"]
        pool, audits["train_vs_validation"], pairs["train_vs_validation"] = remove_train_overlap(train, validation, "clinc_train")
        pool, audits["train_vs_test_after_validation"], pairs["train_vs_test_after_validation"] = remove_train_overlap(pool, test, "clinc_train")

    checks = []
    for a, b, an, bn in [
        (pool, validation, "pool", "validation"),
        (pool, test, "pool", "test"),
        (validation, test, "validation", "test"),
    ]:
        checks.append(verify_disjoint(a, b, an, bn))
    for frame, frame_name in [(pool, "pool"), (validation, "validation"), (test, "test")]:
        assert set(frame["label"]) == expected_labels, f"Unexpected labels in {frame_name}"
        assert frame["source_id"].is_unique, f"Duplicate source_id in {frame_name}"
        assert frame["text_sha256"].equals(frame["text"].map(text_hash)), f"Hash mismatch in {frame_name}"
    source = pd.concat(original_splits.values()).set_index("source_id")
    assert source.index.is_unique, "Duplicate source_id in original source rows"
    for frame in [pool, validation, test]:
        restored = source.loc[frame["source_id"], ["text", "label", "text_sha256"]].reset_index(drop=True)
        pd.testing.assert_frame_equal(restored, frame[["text", "label", "text_sha256"]].reset_index(drop=True))
    # Official protected content and row order must remain complete.
    pd.testing.assert_frame_equal(test, original_splits["test"])
    if dataset_name == "clinc150":
        pd.testing.assert_frame_equal(validation, original_splits["validation"])
    fewshot = sample_fewshot(pool, shot_budget, seed, sampling_mode)
    if shot_budget is not None:
        counts = fewshot.groupby("label").size()
        assert len(counts) == expected_classes and counts.eq(shot_budget).all(), (
            "Few-shot budget/class coverage mismatch")
    assert fewshot["source_id"].is_unique
    pool_by_id = pool.set_index("source_id")
    restored_fewshot = pool_by_id.loc[
        fewshot["source_id"], ["text", "label", "text_sha256"]
    ].reset_index(drop=True)
    pd.testing.assert_frame_equal(
        restored_fewshot,
        fewshot[["text", "label", "text_sha256"]].reset_index(drop=True),
    )
    meta = {
        "protocol_version": "2.0",
        "dataset": dataset_name, "shot_budget": shot_budget, "seed": seed,
        "sampling_mode": sampling_mode,
        "nested_max_shot": 16 if sampling_mode == "nested" else None,
        "validation_fraction": validation_fraction if dataset_name == "banking77" else None,
        "normalization": NORMALIZATION, "hash_algorithm": "SHA256 of normalized UTF-8 text",
        "split_sizes": {"pool": len(pool), "validation": len(validation), "test": len(test),
                        "fewshot": len(fewshot) if shot_budget is not None else None},
        "fewshot_note": "When shot_budget is None, fewshot is a pool copy for compatibility.",
        "classes": expected_classes, "overlap_audits": audits,
        "disjoint_checks": checks, "oos_audit": oos_audit,
        "clinc_original_sha256": original_sha256,
        "id_to_name": id_to_name,
        "source_id_order": "HF original split rows" if dataset_name == "banking77" else "Original CLINC JSON in-domain split rows (legacy protocol)",
    }
    return {"pool": pool, "validation": validation, "test": test, "fewshot": fewshot,
            "protocol_meta": meta, "audit_pairs": pairs}


def validate_training_protocol(protocol):
    """Verify selected training IDs and content using the current seed and budget."""
    meta = protocol["protocol_meta"]
    offer = protocol["pool"]
    selected = protocol["fewshot"]
    budget = meta["shot_budget"]
    mode = meta["sampling_mode"]
    seed = meta["seed"]
    if mode not in {"independent", "nested"}:
        raise ValueError("sampling_mode must be 'independent' or 'nested'")
    print(
        "Protocol metadata:",
        {"protocol_version": meta.get("protocol_version"),
         "sampling_mode": mode,
         "nested_max_shot": meta.get("nested_max_shot"),
         "split_sizes": meta.get("split_sizes")},
    )
    for name in ("pool", "validation", "test", "fewshot"):
        frame = protocol[name]
        assert frame["source_id"].is_unique, f"Duplicate source_id in {name}"
        assert frame["text_sha256"].equals(frame["text"].map(text_hash)), f"Hash mismatch in {name}"

    for left_name, right_name in [
        ("pool", "validation"), ("pool", "test"), ("validation", "test")
    ]:
        verify_disjoint(protocol[left_name], protocol[right_name], left_name, right_name)

    offer_by_id = offer.set_index("source_id")
    expected_hashes = selected["source_id"].map(offer_by_id["text_sha256"])
    assert expected_hashes.notna().all(), "Few-shot ID outside pool"
    assert selected["text_sha256"].reset_index(drop=True).equals(
        expected_hashes.reset_index(drop=True)), "Few-shot hash differs from pool"
    restored_selected = offer_by_id.loc[
        selected["source_id"], ["text", "label"]
    ].reset_index(drop=True)
    pd.testing.assert_frame_equal(
        restored_selected, selected[["text", "label"]].reset_index(drop=True)
    )

    if budget is None:
        pd.testing.assert_frame_equal(selected, offer)
    else:
        expected = sample_fewshot(offer, budget, seed, mode)
        assert set(selected["source_id"]) == set(expected["source_id"]), (
            "Few-shot IDs differ from current seed/budget/mode")
    return True


def reset_random_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def encode_labels(protocol, train, evaluation):
    """Encode labels in a stable contiguous space derived from the full pool."""
    original_labels = sorted(protocol["pool"]["label"].unique().tolist())
    original_to_class = {label: index for index, label in enumerate(original_labels)}
    train_labels = train["label"].map(original_to_class)
    actual = evaluation["label"].map(original_to_class)
    if train_labels.isna().any() or actual.isna().any():
        raise ValueError("Train/evaluation labels must be represented in the pool")
    train_labels = train_labels.to_numpy(dtype=np.int64)
    actual = actual.to_numpy(dtype=np.int64)
    n_classes = len(original_labels)
    if (not original_labels or len(np.unique(train_labels)) != n_classes or
            np.any((train_labels < 0) | (train_labels >= n_classes)) or
            np.any((actual < 0) | (actual >= n_classes))):
        raise ValueError("Encoded train/evaluation labels are incomplete or out of range")
    return original_labels, original_to_class, train_labels, actual


def save_final_test_diagnostics(predictions, class_names, run_tag,
                                evaluation_split, final_test, top_n=20):
    """Display full-label diagnostics only for explicitly authorized final test."""
    if evaluation_split != "test" or final_test is not True:
        return None
    if top_n < 0:
        raise ValueError("top_n must be non-negative")

    n_classes = len(class_names)
    actual = predictions["class_id"].to_numpy(dtype=np.int64)
    predicted = predictions["predicted_class_id"].to_numpy(dtype=np.int64)
    if (np.any((actual < 0) | (actual >= n_classes)) or
            np.any((predicted < 0) | (predicted >= n_classes))):
        raise ValueError("Diagnostic class IDs are outside the supplied class list")

    confusion = pd.crosstab(
        pd.Categorical(predictions["true_intent"], categories=class_names),
        pd.Categorical(predictions["predicted_intent"], categories=class_names),
        dropna=False,
    ).reindex(index=class_names, columns=class_names, fill_value=0)
    assert int(confusion.to_numpy().sum()) == len(predictions), (
        "Confusion matrix omitted rows; check label-to-name mapping")
    confusion.index.name = "true_intent"

    matrix = np.zeros((n_classes, n_classes), dtype=np.int64)
    np.add.at(matrix, (actual, predicted), 1)
    per_class_rows = []
    for class_id, class_name in enumerate(class_names):
        true_positive = int(matrix[class_id, class_id])
        predicted_count = int(matrix[:, class_id].sum())
        support = int(matrix[class_id, :].sum())
        precision = true_positive / predicted_count if predicted_count else 0.0
        recall = true_positive / support if support else 0.0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
        per_class_rows.append({
            "class_id": class_id, "intent": class_name,
            "precision": precision, "recall": recall, "f1": f1,
            "support": support,
        })
    per_class = pd.DataFrame(per_class_rows, columns=[
        "class_id", "intent", "precision", "recall", "f1", "support",
    ])

    error_rows = predictions.loc[predictions["class_id"] != predictions["predicted_class_id"]]
    grouped_rows = []
    for (true_intent, predicted_intent), group in error_rows.groupby(
            ["true_intent", "predicted_intent"], sort=False, dropna=False):
        grouped_rows.append({
            "true_intent": true_intent,
            "predicted_intent": predicted_intent,
            "error_count": len(group),
            "example_texts": json.dumps(group["text"].head(3).tolist(), ensure_ascii=False),
        })
    grouped = pd.DataFrame(grouped_rows, columns=[
        "true_intent", "predicted_intent", "error_count", "example_texts",
    ])
    if not grouped.empty:
        grouped = grouped.sort_values(
            ["error_count", "true_intent", "predicted_intent"],
            ascending=[False, True, True], kind="stable").reset_index(drop=True)
    top_errors = grouped.head(top_n)

    display(confusion)
    display(per_class)
    display(top_errors.head(10))
    return {
        "n_classes": int(n_classes),
        "evaluation_rows": int(len(predictions)),
        "error_count": int(len(error_rows)),
        "error_groups": int(len(grouped)),
        "top_error_groups": int(len(top_errors)),
    }


def run_experiment(protocol, evaluation_split="validation", final_test=False):
    """Train the P0 SetFit model and evaluate exactly one authorized split."""
    if evaluation_split not in {"validation", "test"}:
        raise ValueError("evaluation_split must be 'validation' or 'test'")
    if evaluation_split == "test" and final_test is not True:
        raise ValueError("Test evaluation requires final_test=True")

    validate_training_protocol(protocol)
    meta = protocol["protocol_meta"]
    seed = meta["seed"]
    reset_random_seeds(seed)
    dataset = meta["dataset"]
    budget = meta["shot_budget"]
    shot_tag = "pool" if budget is None else f"{budget}shot"
    train = protocol["pool"] if budget is None else protocol["fewshot"]
    evaluation = protocol[evaluation_split]
    id_to_name = dict(meta["id_to_name"])

    # SetFit expects contiguous classifier labels. Preserve original labels for
    # predictions and name lookup, but map both datasets through the pool's
    # sorted label set so the classifier encoding is identical across datasets.
    original_labels, original_to_class, train_labels, actual = encode_labels(
        protocol, train, evaluation)
    n_classes = len(original_labels)
    if np.any((train_labels < 0) | (train_labels >= n_classes)) or np.any((actual < 0) | (actual >= n_classes)):
        raise ValueError("Encoded train/evaluation labels are out of range")

    train_dataset = Dataset.from_dict({
        "text": train["text"].tolist(),
        "label": train_labels.tolist(),
    })
    print("Preparing model:", BACKBONE, "condition:", shot_tag, flush=True)
    model = SetFitModel.from_pretrained(BACKBONE)
    sampling_mode = meta.get("sampling_mode", "independent")
    run_tag = f"{dataset}_setfit_{shot_tag}_{sampling_mode}_{seed}_{evaluation_split}"
    with TemporaryDirectory(prefix="comp4153_setfit_") as training_directory:
        args = TrainingArguments(
            output_dir=training_directory,
            batch_size=BATCH_SIZE,
            num_epochs=NUM_EPOCHS,
            num_iterations=NUM_ITERATIONS,
            body_learning_rate=BODY_LEARNING_RATE,
            seed=seed,
            report_to="none",
            save_strategy="no",
        )
        if torch.cuda.is_available():
            torch.cuda.synchronize()
            torch.cuda.reset_peak_memory_stats()
        started = time.perf_counter()
        trainer = Trainer(model=model, args=args, train_dataset=train_dataset)
        trainer.train()
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        training_seconds = time.perf_counter() - started
        training_peak = (torch.cuda.max_memory_allocated() / 1024**2
                         if torch.cuda.is_available() else None)

        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()
            torch.cuda.synchronize()
        started = time.perf_counter()
        predicted = model.predict(evaluation["text"].tolist())
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        prediction_seconds = time.perf_counter() - started
        prediction_peak = (torch.cuda.max_memory_allocated() / 1024**2
                           if torch.cuda.is_available() else None)
        predicted = predicted.detach().cpu().tolist() if isinstance(predicted, torch.Tensor) else list(predicted)
        predicted_numeric = np.asarray(predicted)
        if (not np.issubdtype(predicted_numeric.dtype, np.number) or
                not np.isfinite(predicted_numeric).all() or
                not np.equal(predicted_numeric, np.floor(predicted_numeric)).all()):
            raise ValueError("Predictions must be finite integer class IDs")
        predicted = predicted_numeric.astype(np.int64)
        if len(predicted) != len(evaluation):
            raise ValueError("Prediction count does not match evaluation rows")
        if np.any((predicted < 0) | (predicted >= n_classes)):
            raise ValueError("Predicted class IDs are outside the valid contiguous range")

        accuracy = accuracy_score(actual, predicted)
        macro_f1 = f1_score(actual, predicted, average="macro", zero_division=0)
        class_to_original = {index: label for label, index in original_to_class.items()}
        actual_original = [class_to_original[int(label)] for label in actual]
        predicted_original = [class_to_original[int(label)] for label in predicted]
        predictions = evaluation[["source_id", "text", "text_sha256"]].copy().reset_index(drop=True)
        predictions["label"] = actual_original
        predictions["predicted_label"] = predicted_original
        predictions["class_id"] = actual.astype(np.int64, copy=False)
        predictions["predicted_class_id"] = predicted.astype(np.int64, copy=False)
        name_to_label = {name: name for name in id_to_name.values()}
        predictions["true_intent"] = predictions["label"].map(
            lambda label: id_to_name[label] if label in id_to_name else name_to_label.get(label))
        predictions["predicted_intent"] = predictions["predicted_label"].map(
            lambda label: id_to_name[label] if label in id_to_name else name_to_label.get(label))
        class_names = [id_to_name[label] if label in id_to_name else label
                       for label in original_labels]
        errors = predictions.loc[predictions["label"] != predictions["predicted_label"]].copy()
        training_manifest_sha256 = hashlib.sha256(
            train[["source_id", "label", "text_sha256"]].to_csv(index=False).encode("utf-8")
        ).hexdigest()

        settings = {
            "backbone": BACKBONE,
            "batch_size": BATCH_SIZE,
            "num_epochs": NUM_EPOCHS,
            "num_iterations": NUM_ITERATIONS,
            "body_learning_rate": BODY_LEARNING_RATE,
            "max_seq_length": model.model_body.max_seq_length,
            "resource_adjustment": None,
        }
        summary = {
            "method": "setfit",
            "dataset": dataset,
            "shot_budget": budget,
            "shots": budget,
            "shot_tag": shot_tag,
            "sampling_mode": sampling_mode,
            "sampling_seed": seed,
            "training_seed": seed,
            "evaluation_split": evaluation_split,
            "final_test_authorized": evaluation_split == "test" and final_test is True,
            "validation_size": len(protocol["validation"]),
            "test_size": len(protocol["test"]),
            "train_size": len(train),
            "training_manifest_sha256": training_manifest_sha256,
            "evaluation_size": len(evaluation),
            "accuracy": accuracy,
            "macro_f1": macro_f1,
            "error_count": len(errors),
            "validation_accuracy": accuracy if evaluation_split == "validation" else None,
            "validation_macro_f1": macro_f1 if evaluation_split == "validation" else None,
            "test_accuracy": accuracy if evaluation_split == "test" else None,
            "test_macro_f1": macro_f1 if evaluation_split == "test" else None,
            "train_seconds": training_seconds,
            "prediction_seconds": prediction_seconds,
            "train_peak_allocated_mib": training_peak,
            "prediction_peak_allocated_mib": prediction_peak,
            "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
            "training_settings": settings,
            "versions": VERSIONS,
            "protocol_meta": meta,
        }
        if evaluation_split == "test" and final_test is True:
            summary["diagnostics"] = save_final_test_diagnostics(
                predictions, class_names, run_tag, evaluation_split, final_test)
        summary["predictions"] = predictions
        summary["errors"] = errors

        scalar_fields = [
            "method", "dataset", "shot_budget", "sampling_mode",
            "sampling_seed", "training_seed", "evaluation_split",
            "train_size", "validation_size", "test_size", "evaluation_size",
            "validation_accuracy", "validation_macro_f1", "test_accuracy", "test_macro_f1",
            "accuracy", "macro_f1", "error_count", "train_seconds", "prediction_seconds",
            "train_peak_allocated_mib", "prediction_peak_allocated_mib", "gpu",
            "final_test_authorized",
        ]
        display(pd.DataFrame([{field: summary[field] for field in scalar_fields}]))
        display(predictions.head(10))
        display(errors[["text", "true_intent", "predicted_intent"]].head(10))

        parameter_record = {
            "sampling_seed": summary["sampling_seed"],
            "training_seed": summary["training_seed"],
            "evaluation_split": summary["evaluation_split"],
            "training_settings": summary["training_settings"],
            "versions": summary["versions"],
            "protocol_meta": summary["protocol_meta"],
        }
        print(json.dumps(parameter_record, indent=2, ensure_ascii=False))
        print(f"{evaluation_split.title()} accuracy: {accuracy:.4f}; macro-F1: {macro_f1:.4f}; errors: {len(errors)}")
        return summary


def summarize_seed_results(results):
    """Aggregate validation runs while preserving budget and sampling mode."""
    if not results:
        raise ValueError("At least one validation result is required")

    group_fields = (
        "dataset", "method", "shot_budget", "sampling_mode", "evaluation_split",
    )
    grouped = {}
    for result in results:
        if result.get("evaluation_split") != "validation":
            raise ValueError("Seed summaries accept validation results only")
        missing = [field for field in group_fields if field not in result]
        if missing:
            raise ValueError(f"Missing grouping fields: {missing}")
        key = tuple(result[field] for field in group_fields)
        grouped.setdefault(key, []).append(result)

    rows = []
    for key, runs in grouped.items():
        dataset, method, shot_budget, sampling_mode, evaluation_split = key
        seed_pairs = []
        for run in runs:
            if "sampling_seed" not in run or "training_seed" not in run:
                raise ValueError(f"Missing seeds in validation group {key}")
            sampling_seed = run["sampling_seed"]
            training_seed = run["training_seed"]
            if sampling_seed != training_seed:
                raise ValueError(
                    f"Sampling and training seeds differ in group {key}: "
                    f"{sampling_seed} != {training_seed}")
            seed_pairs.append((sampling_seed, training_seed))
        if len(seed_pairs) != len(set(seed_pairs)):
            raise ValueError(f"Duplicate seed run in validation group {key}")
        seeds = sorted({sampling_seed for sampling_seed, _ in seed_pairs})
        if len(seeds) < 3:
            raise ValueError(
                f"Validation group {key} requires at least 3 distinct seeds; got {len(seeds)}")

        shot_tag = "pool" if shot_budget is None else f"{shot_budget}shot"
        run_ids = [
            f"{dataset}_{method}_{shot_tag}_{sampling_mode}_{run['sampling_seed']}_validation"
            for run in runs
        ]
        if len(set(run_ids)) != len(run_ids):
            raise ValueError(f"Duplicate run ID in validation group {key}")

        row = {
            "dataset": dataset,
            "method": method,
            "shot_budget": shot_budget,
            "sampling_mode": sampling_mode,
            "evaluation_split": evaluation_split,
            "n_seeds": len(seeds),
            "seeds": seeds,
            "run_ids": run_ids,
        }
        for metric in (
            "validation_accuracy", "validation_macro_f1",
            "train_seconds", "prediction_seconds",
        ):
            values = pd.to_numeric(
                pd.Series([run.get(metric) for run in runs]), errors="coerce")
            if values.isna().any():
                raise ValueError(f"Missing or invalid {metric} in validation group {key}")
            row[f"{metric}_mean"] = float(values.mean())
            row[f"{metric}_std"] = float(values.std(ddof=1))
        row["variation_note"] = (
            "Seed-to-seed variation includes the training and validation split "
            "for BANKING77; for CLINC150 the protocol split is fixed and the "
            "sampling/training seed changes the few-shot selection and model randomness."
        )
        rows.append(row)

    if not rows:
        raise ValueError("No complete validation groups were supplied")
    summary = pd.DataFrame(rows)
    return summary


{'torch': '2.14.1+cu132', 'setfit': '1.2.0', 'sentence-transformers': '6.2.0', 'transformers': '5.22.1', 'datasets': '5.1.0', 'scikit-learn': '1.9.1'}


## 重建并核验 BANKING77 协议

保留 Source ID、哈希、标签映射及划分无交叉的硬检查。单元中展示数据规模与元数据，不依赖 results/。默认 Full Pool；可选低资源分析保持原有规则。

In [2]:
protocol = build_protocol(DATASET, shot_budget=SHOT_BUDGET, seed=SEED,
                          validation_fraction=VALIDATION_FRACTION, sampling_mode=SAMPLING_MODE)
train = protocol["pool"] if SHOT_BUDGET is None else protocol["fewshot"]
meta = protocol["protocol_meta"]
validate_training_protocol(protocol)
print(json.dumps(meta, ensure_ascii=False))
counts = train["label"].value_counts()
print("Train rows:", len(train), "classes:", len(counts),
      "per-class min/max:", counts.min(), counts.max(),
      "validation:", len(protocol["validation"]), "test:", len(protocol["test"]))
print("CHECK: build_protocol and validate_training_protocol hard checks passed.")
if not torch.cuda.is_available():
    raise RuntimeError("CUDA required for the current GPU experiment")
reset_seed = SEED
random.seed(reset_seed)
np.random.seed(reset_seed)
torch.manual_seed(reset_seed)
torch.cuda.manual_seed_all(reset_seed)
x = torch.randn(64, 64, device="cuda", requires_grad=True)
(x @ x).sum().backward()
torch.cuda.synchronize()
del x
print("GPU forward/backward passed:", torch.cuda.get_device_name(0))


banking_train removed training rows: 7 conflicting pairs: 0
banking_pool removed training rows: 0 conflicting pairs: 0
{"protocol_version": "2.0", "dataset": "banking77", "shot_budget": null, "seed": 42, "sampling_mode": "independent", "nested_max_shot": null, "validation_fraction": 0.1, "normalization": "strip/collapse whitespace and Unicode casefold; punctuation retained", "hash_algorithm": "SHA256 of normalized UTF-8 text", "split_sizes": {"pool": 8996, "validation": 1000, "test": 3080, "fewshot": null}, "fewshot_note": "When shot_budget is None, fewshot is a pool copy for compatibility.", "classes": 77, "overlap_audits": {"train_vs_test": {"left": "banking_train", "right": "protected", "matching_pairs": 7, "unique_texts": 7, "left_rows": 7, "right_rows": 7, "conflicting_pairs": 0}, "pool_vs_validation": {"left": "banking_pool", "right": "protected", "matching_pairs": 0, "unique_texts": 0, "left_rows": 0, "right_rows": 0, "conflicting_pairs": 0}}, "disjoint_checks": [{"left": "pool"

## 训练并展示 Validation 结果

分别记录训练与预测耗时、GPU 峰值显存，展示 Accuracy、Macro-F1、参数、依赖版本以及带真实/预测意图名称的错误示例。完整表保留在内存 summary 中，不写结果文件。

**Final test evaluation, do not use for model selection.** 只有显式设置 EVALUATION_SPLIT="test" 且 FINAL_TEST=True 才评估 Test 并展示混淆矩阵、逐类 F1 和主要混淆意图。最终 Test 实验及其输出需要自行运行并保留。

In [3]:
summary = run_experiment(protocol, evaluation_split=EVALUATION_SPLIT, final_test=FINAL_TEST)
summary_fields = [
    "dataset", "shot_budget", "sampling_mode", "evaluation_split",
    "train_size", "validation_size", "test_size",
    "validation_accuracy", "validation_macro_f1", "test_accuracy", "test_macro_f1",
]
display(pd.DataFrame([{field: summary[field] for field in summary_fields}]))


Preparing model: sentence-transformers/all-MiniLM-L6-v2 condition: pool
{'embedding_loss': '0.1637', 'grad_norm': '1.131', 'learning_rate': '0', 'epoch': '0.0001778'}
{'embedding_loss': '0.1452', 'grad_norm': '0.9543', 'learning_rate': '1.741e-06', 'epoch': '0.008892'}
{'embedding_loss': '0.1415', 'grad_norm': '0.8187', 'learning_rate': '3.517e-06', 'epoch': '0.01778'}
{'embedding_loss': '0.1387', 'grad_norm': '1.036', 'learning_rate': '5.293e-06', 'epoch': '0.02668'}
{'embedding_loss': '0.1203', 'grad_norm': '0.5844', 'learning_rate': '7.069e-06', 'epoch': '0.03557'}
{'embedding_loss': '0.1077', 'grad_norm': '0.7012', 'learning_rate': '8.845e-06', 'epoch': '0.04446'}
{'embedding_loss': '0.104', 'grad_norm': '0.5983', 'learning_rate': '1.062e-05', 'epoch': '0.05335'}
{'embedding_loss': '0.09607', 'grad_norm': '0.7967', 'learning_rate': '1.24e-05', 'epoch': '0.06224'}
{'embedding_loss': '0.09821', 'grad_norm': '0.6625', 'learning_rate': '1.417e-05', 'epoch': '0.07114'}
{'embedding_loss'

100%|##########| 5623/5623 [06:50<00:00, 13.69it/s]


,dataset,shot_budget,sampling_mode,evaluation_split,train_size,validation_size,test_size,validation_accuracy,validation_macro_f1,test_accuracy,test_macro_f1
0,banking77,None,independent,validation,8996,1000,3080,0.865,0.856228,None,None


## P2 · Optional controlled low-resource analysis

`RUN_SHOT_GRID=False` 默认关闭。启用后在 Validation 比较 4/8/16-shot，并显式传入 `SAMPLING_MODE`。`nested` 模式下 4/8/16 使用相同逐类随机排列的前缀；`independent` 模式下各预算独立抽样。


In [4]:
if RUN_SHOT_GRID:
    shot_results = []
    for budget in [4, 8, 16]:
        condition = build_protocol(DATASET, shot_budget=budget, seed=SEED,
                                   validation_fraction=VALIDATION_FRACTION,
                                   sampling_mode=SAMPLING_MODE)
        shot_results.append(run_experiment(condition, evaluation_split="validation", final_test=False))
    display(pd.DataFrame(shot_results)[["shot_budget", "train_size", "validation_accuracy", "validation_macro_f1"]])
else:
    print("RUN_SHOT_GRID=False; optional analysis disabled.")


RUN_SHOT_GRID=False; optional analysis disabled.


## 可选：多随机种子受控分析

RUN_MULTI_SEED=False 默认关闭。启用后仅针对 BANKING77，沿用 8-shot、independent/nested 和 seeds 42/43/44。Mean ± Std 表直接显示，以 run_ids 关联本次内存结果，不生成文件。

In [5]:
if RUN_MULTI_SEED:
    repeated = []
    for dataset_name in ANALYSIS_DATASETS:
        for sampling_mode in ANALYSIS_SAMPLING_MODES:
            for seed in SEEDS:
                condition = build_protocol(dataset_name, shot_budget=8, seed=seed,
                                           validation_fraction=VALIDATION_FRACTION,
                                           sampling_mode=sampling_mode)
                repeated.append(run_experiment(condition, evaluation_split="validation", final_test=False))
    seed_summary = summarize_seed_results(repeated)
    display(seed_summary)
else:
    print("RUN_MULTI_SEED=False; repeated-seed controlled analysis disabled.")


RUN_MULTI_SEED=False; repeated-seed controlled analysis disabled.
